# F1 Dataset - Data Pre-processing
Dataset: https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020


In [2]:
import pandas as pd
import numpy as np


results     = pd.read_csv('results.csv')
drivers     = pd.read_csv('drivers.csv')
races       = pd.read_csv('races.csv')
constructors = pd.read_csv('constructors.csv')
lap_times   = pd.read_csv('lap_times.csv')




## Questão 1
**Juntar as tabelas results e drivers pelo driverId, filtrar as linhas onde nationality == 'British' e calcular a média de pontos.**

- `x` = `points` (results)
- `y` = `nationality` (drivers) — valor escolhido: **'British'**

In [3]:
# Merge results com drivers
results_nationality = results.merge(drivers[['driverId', 'nationality']], on='driverId')

# Filtrar pela nacionalidade escolhida
nacionalidade = 'Portuguese'
avg_points = results_nationality[results_nationality['nationality'] == nacionalidade]['points'].mean()

print(f"Média de pontos para pilotos '{nacionalidade}': {avg_points:.4f}")

Média de pontos para pilotos 'Portuguese': 0.0920


## Questão 2
**Medida robusta de variabilidade de `x`, por valores de uma variável discreta `z` (de tabela diferente de `x` e diferente de `y`).**

- `x` = `points` (results)
- `z` = `circuitId` via `races` (tabela diferente de `x` e diferente de `drivers`)
- Medida robusta: **Desvio Padrão**

In [4]:
# Merge results com races para obter circuitId
results_circuits = results.merge(races[['raceId', 'circuitId']], on='raceId')

# IQR dos pontos por circuito
desvio_padrao = results_circuits.groupby('circuitId')['points'].std().reset_index()
desvio_padrao.columns = ['circuitId', 'Desvio_Padrão']

print(desvio_padrao.to_string(index=False))

 circuitId  Desvio_Padrão
         1       5.539017
         2       5.237123
         3       6.371374
         4       5.296297
         5       5.419446
         6       3.920894
         7       4.464650
         8       2.590694
         9       4.306098
        10       3.631545
        11       4.954671
        12       5.874759
        13       4.330675
        14       3.871910
        15       6.752664
        16       2.633030
        17       6.109985
        18       4.790870
        19       2.186795
        20       3.060664
        21       3.453184
        22       4.945269
        24       7.714845
        25       2.274904
        26       2.121235
        27       2.171895
        28       2.343117
        29       2.089954
        30       2.216039
        31       2.366432
        32       4.873580
        33       1.949164
        34       3.850225
        35       6.893709
        36       2.073925
        37       2.139216
        38       2.153904
        39  

## Questão 3
**Cruzar a nacionalidade do piloto com a nacionalidade do construtor.**

- `a` = `nationality` (drivers)
- `b` = `constructorId` via constructors → `nationality` do construtor

In [5]:
# Merge results com drivers e constructors
results_drivers_constructors = results.merge(drivers[['driverId', 'nationality']], on='driverId')
results_drivers_constructors = results_drivers_constructors.merge(constructors[['constructorId', 'nationality']], on='constructorId', suffixes=('_driver', '_constructor'))

# Tabela de contingência
contingencia = pd.crosstab(results_drivers_constructors['nationality_driver'], results_drivers_constructors['nationality_constructor'])

print("Tabela de contingência")
print(contingencia)

Tabela de contingência
nationality_constructor  American  Australian  Austrian  Belgian  Brazilian  \
nationality_driver                                                            
American                      468           0         0        0          0   
American-Italian                0           0         0        0          0   
Argentine                       0           0         0        0          0   
Argentine-Italian               0           0         0        0          0   
Argentinian                     0           0         0        0          0   
Australian                      0           1       229        0          0   
Austrian                        3           0        30        0          0   
Belgian                         0           0         0        1          0   
Brazilian                       2           0         0        0        126   
British                        19           0        72        0          0   
Canadian                     

## Questão 4
**Total de pontos por piloto em cada corrida e máximo entre pilotos, por nacionalidade do piloto.**

- Passo 1: soma de `points` por `driverId` 
- Passo 2: máximo entre pilotos, agrupado por `nationality`

In [8]:
# Soma de pontos por piloto
points_per_driver = results.groupby('driverId')['points'].sum().reset_index()
points_per_driver.columns = ['driverId', 'total_points']

# Merge com drivers para obter nacionalidade
points_per_driver = points_per_driver.merge(drivers[['driverId', 'nationality']], on='driverId')

# Máximo de pontos totais por nacionalidade
nacionalidade_mais_pontos = points_per_driver.groupby('nationality')['total_points'].max().reset_index()
nacionalidade_mais_pontos.columns = ['nationality', 'max_total_points']
nacionalidade_mais_pontos = nacionalidade_mais_pontos.sort_values('max_total_points', ascending=False)

print(nacionalidade_mais_pontos.to_string(index=False))

      nationality  max_total_points
          British            4820.5
           German            3098.0
            Dutch            2912.5
          Spanish            2329.0
          Finnish            1873.0
          Mexican            1585.0
       Monegasque            1363.0
       Australian            1320.0
        Brazilian            1167.0
           French             798.5
         Austrian             420.5
        Argentine             310.0
        Colombian             307.0
         Canadian             286.0
          Italian             281.0
           Polish             274.0
    South African             255.0
    New Zealander             248.0
             Thai             238.0
            Swiss             212.0
          Swedish             206.0
          Russian             202.0
           Danish             196.0
          Belgian             181.0
         American             180.0
         Japanese             125.0
       Venezuelan           

## Questão 5
**Forma Standard de uma variável de texto: converter para minúsculas e remover espaços exteriores e pontuação.**

- Variável: `surname` dos pilotos (drivers)

In [61]:
import re

def standardize(text):
    if pd.isna(text):
        return text
    text = text.lower()           
    text = text.strip()           
    text = re.sub(r'[^\w\s]', '', text) 
    return text

drivers['surname_clean'] = drivers['surname'].apply(standardize)

print(drivers[['surname', 'surname_clean']].to_string(index=False))

                surname          surname_clean
               Hamilton               hamilton
               Heidfeld               heidfeld
                Rosberg                rosberg
                 Alonso                 alonso
             Kovalainen             kovalainen
               Nakajima               nakajima
               Bourdais               bourdais
              Räikkönen              räikkönen
                 Kubica                 kubica
                  Glock                  glock
                   Sato                   sato
             Piquet Jr.              piquet jr
                  Massa                  massa
              Coulthard              coulthard
                 Trulli                 trulli
                  Sutil                  sutil
                 Webber                 webber
                 Button                 button
               Davidson               davidson
                 Vettel                 vettel
             